# 1. N-gram Modeling

Example : 
- "I love machine"
- "I love Python"
- "I enjoy coding"

### Bigram Counts

| Bigram          | Count |
|-----------------|-------|
| (I, love)       | 2     |
| (I, enjoy)      | 1     |
| (love, machine) | 1     |
| (love, Python)  | 1     |
| (enjoy, coding) | 1     |

### Context Counts (first word of bigram)

| Word (context) | Count |
|----------------|-------|
| I              | 3     |
| love           | 2     |
| enjoy          | 1     |

---

## 3. Compute MLE Probabilities


$P(\text{love} \mid I) = \frac{2}{3} \approx 0.667$

$P(\text{enjoy} \mid I) = \frac{1}{3} \approx 0.333$

$P(\text{machine} \mid love) = \frac{1}{2} = 0.5$

$P(\text{Python} \mid love) = \frac{1}{2} = 0.5$
 

---

## 4. Explanation

- To predict the next word after a given word, look at **all bigrams that start with that word**.  
- Divide the count of each bigram by the total count of the context word.  
- The word with the **highest probability** is the predicted next word.  

**Example:**  
- Context = `"I"` → candidates `"love"` (0.667) and `"enjoy"` (0.333) → predicte


In [1]:
text = 'artificial intelligence improves data analysis, artificial intelligence powers modern applicatons.'

In [2]:
from nltk.tokenize import word_tokenize

tokens = [t.lower() for t in word_tokenize(text)]

In [3]:
from nltk.util import ngrams
from collections import Counter

bigrams = list(ngrams(tokens, 2))
trigrams = list(ngrams(tokens, 3))

bigram_freq = Counter(bigrams)
trigram_freq = Counter(trigrams)

In [4]:
bigram_context_counts = Counter([bg[0] for bg in bigrams])
trigram_context_counts = Counter([tg[:2] for tg in trigrams])

bigram_mle = {bg: count / bigram_context_counts[bg[0]] for bg, count in bigram_freq.items()}
trigram_mle = {tg: count / trigram_context_counts[tg[:2]] for tg, count in trigram_freq.items()}

print("Bigram MLE probabilities:")
for k, v in list(bigram_mle.items()):
    print(k, ":", v)

print("\nTrigram MLE probabilities:")
for k, v in list(trigram_mle.items()):
    print(k, ":", v)

Bigram MLE probabilities:
('artificial', 'intelligence') : 1.0
('intelligence', 'improves') : 0.5
('improves', 'data') : 1.0
('data', 'analysis') : 1.0
('analysis', ',') : 1.0
(',', 'artificial') : 1.0
('intelligence', 'powers') : 0.5
('powers', 'modern') : 1.0
('modern', 'applicatons') : 1.0
('applicatons', '.') : 1.0

Trigram MLE probabilities:
('artificial', 'intelligence', 'improves') : 0.5
('intelligence', 'improves', 'data') : 1.0
('improves', 'data', 'analysis') : 1.0
('data', 'analysis', ',') : 1.0
('analysis', ',', 'artificial') : 1.0
(',', 'artificial', 'intelligence') : 1.0
('artificial', 'intelligence', 'powers') : 0.5
('intelligence', 'powers', 'modern') : 1.0
('powers', 'modern', 'applicatons') : 1.0
('modern', 'applicatons', '.') : 1.0


In [5]:
def predict_next_word_trigram(context_words, trigram_mle):
    
    candidates = {tg[2]: prob for tg, prob in trigram_mle.items() if tg[:2] == tuple(context_words)}
    next_word = max(candidates, key=candidates.get)
    
    return next_word

c1 = 'artificial'
c2 = 'intelligence'

next_word = predict_next_word_trigram([c1, c2], trigram_mle)
print(f"Next word after '{c1} {c2}':", next_word)

Next word after 'artificial intelligence': improves


# 2. Data Sparsity & Smoothing Techniques: How smoothing changes probability distribution and model behavior

In [7]:
text = 'students study machine learning. Students study Data Science.'

tokens = [t.lower() for t in word_tokenize(text)]

In [26]:
bigram = list(ngrams(tokens, 2))

bigram_freq = Counter(bigram)
bigram_context_freq = Counter(bg[0] for bg in bigram)

for bg, count in bigram_freq.items():
    context_count = bigram_context_freq[bg[0]]
    mle = count / context_count
    print(f"Bigram: {bg}, Count: {count}, Context Count: {context_count}, MLE: {mle}")

Bigram: ('students', 'study'), Count: 2, Context Count: 2, MLE: 1.0
Bigram: ('study', 'machine'), Count: 1, Context Count: 2, MLE: 0.5
Bigram: ('machine', 'learning'), Count: 1, Context Count: 1, MLE: 1.0
Bigram: ('learning', '.'), Count: 1, Context Count: 1, MLE: 1.0
Bigram: ('.', 'students'), Count: 1, Context Count: 1, MLE: 1.0
Bigram: ('study', 'data'), Count: 1, Context Count: 2, MLE: 0.5
Bigram: ('data', 'science'), Count: 1, Context Count: 1, MLE: 1.0
Bigram: ('science', '.'), Count: 1, Context Count: 1, MLE: 1.0


In [30]:
vocab = set(tokens)
v = len(vocab)

for bg, count in bigram_freq.items():
    context_count = bigram_context_freq[bg[0]]
    laplace_prob = (count + 1) / (context_count + v)
    print(f"Bigram: {bg}, Laplace Probability: {laplace_prob:.3f}")

Bigram: ('students', 'study'), Laplace Probability: 0.333
Bigram: ('study', 'machine'), Laplace Probability: 0.222
Bigram: ('machine', 'learning'), Laplace Probability: 0.250
Bigram: ('learning', '.'), Laplace Probability: 0.250
Bigram: ('.', 'students'), Laplace Probability: 0.250
Bigram: ('study', 'data'), Laplace Probability: 0.222
Bigram: ('data', 'science'), Laplace Probability: 0.250
Bigram: ('science', '.'), Laplace Probability: 0.250


In [42]:
test = 'students study ai.'
tokens = word_tokenize(test.lower())
bigram_test = list(ngrams(tokens, 2))

print("MLE\n")
for bg in bigram_test:
    count = bigram_freq.get(bg, 0)
    context_count = bigram_context_freq.get(bg[0], 0)
    if context_count == 0:
        print(f"Bigram: {bg}, MLE: undefined (context unseen)")
        continue
    else:
        mle = count / context_count
        print(f"Bigram: {bg}, MLE: {mle}")
print("="*50)
print("Laplace Smoothing\n")
for bg in bigram_test:
    count = bigram_freq.get(bg, 0)
    context_count = bigram_context_freq.get(bg[0], 0)
    laplace_prob = (count + 1) / (context_count + v)
    print(f"Bigram: {bg}, Laplace Probability: {laplace_prob:.3f}")

MLE

Bigram: ('students', 'study'), MLE: 1.0
Bigram: ('study', 'ai'), MLE: 0.0
Bigram: ('ai', '.'), MLE: undefined (context unseen)
Laplace Smoothing

Bigram: ('students', 'study'), Laplace Probability: 0.333
Bigram: ('study', 'ai'), Laplace Probability: 0.111
Bigram: ('ai', '.'), Laplace Probability: 0.143


# 3. Evaluate language models using Perplexity

In [43]:
from nltk.corpus import brown

sentences = brown.sents()  

for i in range(5):
    print(sentences[i])


['The', 'Fulton', 'County', 'Grand', 'Jury', 'said', 'Friday', 'an', 'investigation', 'of', "Atlanta's", 'recent', 'primary', 'election', 'produced', '``', 'no', 'evidence', "''", 'that', 'any', 'irregularities', 'took', 'place', '.']
['The', 'jury', 'further', 'said', 'in', 'term-end', 'presentments', 'that', 'the', 'City', 'Executive', 'Committee', ',', 'which', 'had', 'over-all', 'charge', 'of', 'the', 'election', ',', '``', 'deserves', 'the', 'praise', 'and', 'thanks', 'of', 'the', 'City', 'of', 'Atlanta', "''", 'for', 'the', 'manner', 'in', 'which', 'the', 'election', 'was', 'conducted', '.']
['The', 'September-October', 'term', 'jury', 'had', 'been', 'charged', 'by', 'Fulton', 'Superior', 'Court', 'Judge', 'Durwood', 'Pye', 'to', 'investigate', 'reports', 'of', 'possible', '``', 'irregularities', "''", 'in', 'the', 'hard-fought', 'primary', 'which', 'was', 'won', 'by', 'Mayor-nominate', 'Ivan', 'Allen', 'Jr.', '.']
['``', 'Only', 'a', 'relative', 'handful', 'of', 'such', 'reports